# MeterMind — Milestone 1: Synthetic Billing Dataset

This notebook creates a reproducible fictional billing environment for MeterMind.

**Goal:** generate customers, versioned pricing, usage events, invoice lines, and known anomalies. We establish expected outcomes before adding LangChain or LangGraph.

**Important:** all records are synthetic. No real customer or payment data is used.

## 1. Project setup

This milestone uses Python's standard library only. We will add dependencies when the billing engine and agent workflow need them.

In [ ]:
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_UP
from typing import Optional
import json
import random

SEED = 42
random.seed(SEED)

print('MeterMind dataset environment ready.')

## 2. Define the data models

We use explicit IDs and timestamps so investigations can cite exact records. Monetary values use `Decimal`, not binary floating-point.

In [ ]:
@dataclass(frozen=True)
class Customer:
    customer_id: str
    name: str
    plan_id: str
    currency: str = 'USD'

@dataclass(frozen=True)
class PricingVersion:
    pricing_version_id: str
    plan_id: str
    unit_price: Decimal
    effective_from: str
    effective_to: Optional[str] = None

@dataclass(frozen=True)
class UsageEvent:
    event_id: str
    customer_id: str
    occurred_at: str
    quantity: int
    metric: str = 'api_request'
    status: str = 'accepted'

@dataclass(frozen=True)
class InvoiceLine:
    invoice_line_id: str
    invoice_id: str
    customer_id: str
    billing_period: str
    quantity: int
    unit_price: Decimal
    amount: Decimal
    pricing_version_id: str

def money(value: Decimal) -> Decimal:
    return value.quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)

print('Data models defined.')

## 3. Create synthetic customers and pricing versions

The initial model is intentionally simple: customers are charged per API request. Pricing versions allow us to test plan changes without confusing the billing engine.

In [ ]:
customers = [
    Customer('CUS-1001', 'Acme AI', 'STARTER'),
    Customer('CUS-1002', 'Northstar Apps', 'GROWTH'),
    Customer('CUS-1003', 'Orbit API', 'PRO'),
    Customer('CUS-1004', 'Pinecone Labs', 'STARTER'),
]

pricing_versions = [
    PricingVersion('PRICE-STARTER-V1', 'STARTER', Decimal('0.0100'), '2026-01-01', '2026-06-30'),
    PricingVersion('PRICE-STARTER-V2', 'STARTER', Decimal('0.0120'), '2026-07-01'),
    PricingVersion('PRICE-GROWTH-V1', 'GROWTH', Decimal('0.0080'), '2026-01-01'),
    PricingVersion('PRICE-PRO-V1', 'PRO', Decimal('0.0050'), '2026-01-01'),
]

customers_by_id = {c.customer_id: c for c in customers}
pricing_by_id = {p.pricing_version_id: p for p in pricing_versions}

print(f'Customers: {len(customers)}')
print(f'Pricing versions: {len(pricing_versions)}')
for p in pricing_versions:
    print(p.pricing_version_id, '| plan=', p.plan_id, '| unit price=$', p.unit_price, '| effective=', p.effective_from, 'to', p.effective_to or 'open-ended')

## 4. Generate a reproducible usage-event ledger

Each customer receives a small batch of usage events for July 2026. The seed ensures the same dataset is generated on each run.

In [ ]:
BILLING_PERIOD = '2026-07'
EVENTS_PER_CUSTOMER = 12

usage_events = []
for customer in customers:
    for i in range(1, EVENTS_PER_CUSTOMER + 1):
        day = ((i - 1) % 28) + 1
        hour = (i * 3) % 24
        occurred_at = f'2026-07-{day:02d}T{hour:02d}:00:00+00:00'
        quantity = random.randint(50, 500)
        usage_events.append(
            UsageEvent(
                event_id=f'EVT-{customer.customer_id[-4:]}-{i:03d}',
                customer_id=customer.customer_id,
                occurred_at=occurred_at,
                quantity=quantity,
            )
        )

print(f'Generated {len(usage_events)} unique usage events.')
print('Sample:', usage_events[0])

## 5. Resolve the correct pricing version

The applicable price is selected using the event timestamp and plan. This is a deterministic business rule, not an LLM decision.

In [ ]:
def pricing_for(plan_id: str, occurred_at: str) -> PricingVersion:
    event_date = occurred_at[:10]
    matches = [
        p for p in pricing_versions
        if p.plan_id == plan_id
        and p.effective_from <= event_date
        and (p.effective_to is None or event_date <= p.effective_to)
    ]
    if len(matches) != 1:
        raise ValueError(f'Expected exactly one pricing version for plan={plan_id}, date={event_date}; found {len(matches)}')
    return matches[0]

for customer in customers:
    p = pricing_for(customer.plan_id, '2026-07-15T12:00:00+00:00')
    print(customer.customer_id, '->', p.pricing_version_id, '@ $', p.unit_price, 'per request')

## 6. Generate baseline invoice lines

For this first milestone, invoice quantities are derived from accepted usage events and grouped by customer and pricing version. In the next milestone, we'll make reconciliation more robust and handle event-level idempotency and adjustments explicitly.

In [ ]:
from collections import defaultdict

def build_baseline_invoice_lines(events: list[UsageEvent]) -> list[InvoiceLine]:
    totals = defaultdict(int)
    for event in events:
        customer = customers_by_id[event.customer_id]
        price = pricing_for(customer.plan_id, event.occurred_at)
        totals[(event.customer_id, price.pricing_version_id)] += event.quantity

    lines = []
    for idx, ((customer_id, pricing_version_id), quantity) in enumerate(sorted(totals.items()), start=1):
        price = pricing_by_id[pricing_version_id]
        lines.append(
            InvoiceLine(
                invoice_line_id=f'LINE-{idx:04d}',
                invoice_id=f'INV-{customer_id[-4:]}-{BILLING_PERIOD}',
                customer_id=customer_id,
                billing_period=BILLING_PERIOD,
                quantity=quantity,
                unit_price=price.unit_price,
                amount=money(price.unit_price * Decimal(quantity)),
                pricing_version_id=pricing_version_id,
            )
        )
    return lines

baseline_invoice_lines = build_baseline_invoice_lines(usage_events)
print(f'Generated {len(baseline_invoice_lines)} baseline invoice lines.')
for line in baseline_invoice_lines:
    print(line)

## 7. Inject known anomalies into a separate observed dataset

We preserve the clean source ledger. The observed data simulates what a flawed production pipeline might contain. Each injected anomaly has a known label for evaluation.

In [ ]:
clean_events = list(usage_events)
observed_events = list(usage_events)
known_anomalies = []

# Anomaly A: duplicate event is delivered twice.
duplicate_source = usage_events[2]
observed_events.append(duplicate_source)
known_anomalies.append({
    'anomaly_id': 'ANOM-001',
    'type': 'duplicate_event',
    'customer_id': duplicate_source.customer_id,
    'event_id': duplicate_source.event_id,
    'expected_detection': True,
    'description': 'The same event ID appears more than once in the observed event stream.'
})

# Anomaly B: a clean event is missing from the observed stream.
missing_source = usage_events[8]
observed_events = [e for e in observed_events if e.event_id != missing_source.event_id]
known_anomalies.append({
    'anomaly_id': 'ANOM-002',
    'type': 'missing_event',
    'customer_id': missing_source.customer_id,
    'event_id': missing_source.event_id,
    'expected_detection': True,
    'description': 'An event present in the source ledger is absent from the observed stream.'
})

# Anomaly C: one invoice line is deliberately undercharged by $5.00.
observed_invoice_lines = list(baseline_invoice_lines)
target_line = observed_invoice_lines[0]
undercharged_line = InvoiceLine(
    invoice_line_id=target_line.invoice_line_id,
    invoice_id=target_line.invoice_id,
    customer_id=target_line.customer_id,
    billing_period=target_line.billing_period,
    quantity=target_line.quantity,
    unit_price=target_line.unit_price,
    amount=money(max(Decimal('0.00'), target_line.amount - Decimal('5.00'))),
    pricing_version_id=target_line.pricing_version_id,
)
observed_invoice_lines[0] = undercharged_line
known_anomalies.append({
    'anomaly_id': 'ANOM-003',
    'type': 'invoice_undercharge',
    'customer_id': target_line.customer_id,
    'invoice_line_id': target_line.invoice_line_id,
    'expected_detection': True,
    'description': 'Recorded invoice amount is $5.00 lower than the baseline expected amount.'
})

print('Clean events:', len(clean_events))
print('Observed events:', len(observed_events))
print('Known anomaly cases:', len(known_anomalies))
for anomaly in known_anomalies:
    print(anomaly['anomaly_id'], anomaly['type'], '|', anomaly['description'])

## 8. Deterministic validation checks

These checks confirm that our test fixture contains the anomalies we intended to create. They do not yet represent the production reconciliation engine.

In [ ]:
from collections import Counter

event_counts = Counter(e.event_id for e in observed_events)
duplicate_ids = sorted(event_id for event_id, count in event_counts.items() if count > 1)
clean_ids = {e.event_id for e in clean_events}
observed_ids = {e.event_id for e in observed_events}
missing_ids = sorted(clean_ids - observed_ids)

assert duplicate_source.event_id in duplicate_ids, 'Expected duplicate event was not found.'
assert missing_source.event_id in missing_ids, 'Expected missing event was not found.'
assert len(known_anomalies) == 3, 'Unexpected number of injected anomaly cases.'
assert undercharged_line.amount == money(target_line.amount - Decimal('5.00')), 'Invoice anomaly amount is incorrect.'

print('Duplicate event IDs:', duplicate_ids)
print('Missing event IDs:', missing_ids)
print('Baseline amount:', target_line.amount)
print('Observed invoice amount:', undercharged_line.amount)
print('Difference:', money(target_line.amount - undercharged_line.amount))
print('\nAll Milestone 1 fixture checks passed.')

## 9. Export fixture data for later milestones

Export JSON snapshots so future notebook sections or application modules can consume the same reproducible fixture. Decimal values are serialized as strings to preserve exact monetary representation.

In [ ]:
def json_safe(value):
    if isinstance(value, Decimal):
        return str(value)
    if hasattr(value, '__dataclass_fields__'):
        return {k: json_safe(v) for k, v in asdict(value).items()}
    if isinstance(value, list):
        return [json_safe(v) for v in value]
    if isinstance(value, dict):
        return {k: json_safe(v) for k, v in value.items()}
    return value

fixture = {
    'billing_period': BILLING_PERIOD,
    'customers': json_safe(customers),
    'pricing_versions': json_safe(pricing_versions),
    'clean_events': json_safe(clean_events),
    'observed_events': json_safe(observed_events),
    'baseline_invoice_lines': json_safe(baseline_invoice_lines),
    'observed_invoice_lines': json_safe(observed_invoice_lines),
    'known_anomalies': json_safe(known_anomalies),
}

with open('metermind_synthetic_fixture.json', 'w', encoding='utf-8') as f:
    json.dump(fixture, f, indent=2)

print('Exported metermind_synthetic_fixture.json to the notebook working directory.')

## Milestone 1 completion checklist

- [x] Define typed data models.
- [x] Create customers and versioned pricing.
- [x] Generate reproducible usage events.
- [x] Generate baseline invoice lines.
- [x] Inject duplicate, missing-event, and undercharge scenarios.
- [x] Validate that the intended anomalies exist.
- [x] Export a JSON fixture.

**Next milestone:** implement the deterministic reconciliation engine and tool functions. Do not add LangGraph agents until these tools pass their own tests.